In [0]:
spark.conf.set("fs.azure.account.auth.type.proddatalakefordatabrick.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.proddatalakefordatabrick.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.proddatalakefordatabrick.dfs.core.windows.net", "8b03b30e-c838-4568-a41e-a801634f0e96")
spark.conf.set("fs.azure.account.oauth2.client.secret.proddatalakefordatabrick.dfs.core.windows.net", dbutils.secrets.get(scope="darshanscope", key="app-secret"))
spark.conf.set("fs.azure.account.oauth2.client.endpoint.proddatalakefordatabrick.dfs.core.windows.net", "https://login.microsoftonline.com/5ff33f54-37cd-4d6d-9156-f6c00f164999/oauth2/token")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

df_silver = spark.read.table("proddatabricksdemo.silver.products")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS proddatabricksdemo.gold.DimProducts (
    DimProductsKey BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id STRING,
    product_name STRING,
    category STRING,
    brand STRING,
    price DOUBLE,
    create_date TIMESTAMP,
    update_date TIMESTAMP
)
USING DELTA;

In [0]:
# Target Gold Delta Table
gold_table = DeltaTable.forName(
    spark,
    "proddatabricksdemo.gold.DimProducts"
)

In [0]:
(
    gold_table.alias("target")
    .merge(
        df_silver.alias("source"),
        "target.product_id = source.product_id"
    )
    .whenMatchedUpdate(
        set={
            "product_name": "source.product_name",
            "category": "source.category",
            "brand": "source.brand",
            "price": "source.price",
            "create_date": "current_timestamp()",
            "update_date": "current_timestamp()"
        }
    )
    .whenNotMatchedInsert(
        values={
            "product_id": "source.product_id",
            "product_name": "source.product_name",
            "category": "source.category",
            "brand": "source.brand",
            "price": "source.price",
            "create_date": "current_timestamp()",
            "update_date": "current_timestamp()"
        }
    )
    .execute()
)

In [0]:
df_final = spark.read.table("proddatabricksdemo.gold.DimProducts")
df_final.display()

In [0]:
df_final.write.format("delta").mode("overwrite").save("abfss://destination@proddatalakefordatabrick.dfs.core.windows.net/gold/products")